# 命名实体识别（BERT + 人民日报语料）

## 项目概览
- **模型**：BERT-Base-Chinese (102M 参数)
- **任务**：命名实体识别 NER (Named Entity Recognition)
- **数据集**：人民日报语料（China People's Daily）
- **实体类型**：LOC（地名）、ORG（机构名）、PER（人名）
- **标注方案**：IOB2（B-开头、I-中间、O-非实体）
- **评估指标**：Precision、Recall、F1（seqeval 实体级别评估）

## NER 的本质是 Token Classification
```
输入: [CLS] 毛 泽 东 在 北 京 [SEP]
标签:   O  B-PER I-PER I-PER O B-LOC I-LOC  O
```
与抽取式 QA 类似，都是对每个 token 做分类，区别在于标签定义不同。

## IOB2 标注规则
- **B-XXX**：实体 XXX 的起始 token
- **I-XXX**：实体 XXX 的内部 token
- **O**：非实体 token

## 运行说明
- 使用快捷键 `Shift + Enter` 逐个运行 Cell
- 或点击菜单 `Cell → Run All` 一次性运行全部
- 整个流程耗时约 1-2 小时（GPU）

## 第一步：环境检查和库导入

In [ ]:
import os
import json
import random
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.nn import CrossEntropyLoss
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoConfig,
    AutoTokenizer,
    BertPreTrainedModel,
    BertModel,
    AdamW,
    get_scheduler
)

# 任务	用哪个 AutoModel
# 文本分类	AutoModelForSequenceClassification
# 命名实体识别	AutoModelForTokenClassification  ← 本项目（等价于此）
# 问答（抽取式）	AutoModelForQuestionAnswering
# 翻译 / 摘要	AutoModelForSeq2SeqLM

from seqeval.metrics import classification_report
from seqeval.scheme import IOB2
from tqdm.auto import tqdm

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("✓ 库导入成功")
print(f"✓ PyTorch 版本: {torch.__version__}")
print(f"✓ GPU 可用: {torch.cuda.is_available()}")
print(f"✓ MPS 可用: {torch.backends.mps.is_available()}")

## 第二步：数据加载

数据格式（BIO 格式，每行一个字和对应标签，句子间用空行分隔）：
```
毛 B-PER
泽 I-PER
东 I-PER
在 O
北 B-LOC
京 I-LOC

新 B-ORG
华 I-ORG
社 I-ORG
```

In [ ]:
DATA_DIR   = '../../data/china-people-daily-ner-corpus/'
OUTPUT_DIR = './ner_model_output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_FILE = os.path.join(DATA_DIR, 'example.train')
DEV_FILE   = os.path.join(DATA_DIR, 'example.dev')
TEST_FILE  = os.path.join(DATA_DIR, 'example.test')

for name, path in [('训练数据', TRAIN_FILE), ('验证数据', DEV_FILE), ('测试数据', TEST_FILE)]:
    print(f"{name}: {path}")
    print(f"  存在: {os.path.exists(path)}")
    if os.path.exists(path):
        print(f"  大小: {os.path.getsize(path) / 1024:.1f} KB")

print(f"\n输出目录: {OUTPUT_DIR}")

## 第三步：标签体系和数据集类

In [ ]:
# 实体类别（3 类）
CATEGORIES = ['LOC', 'ORG', 'PER']

# 构建标签映射：O + B-/I- 各类别，共 7 个标签
id2label = {0: 'O'}
for cat in CATEGORIES:
    id2label[len(id2label)] = f'B-{cat}'
    id2label[len(id2label)] = f'I-{cat}'
label2id = {v: k for k, v in id2label.items()}

print(f"共 {len(id2label)} 个标签:")
for k, v in id2label.items():
    print(f"  {k}: {v}")


class PeopleDaily(Dataset):
    """人民日报 NER 数据集（BIO 格式）"""

    def __init__(self, data_file):
        self.data = self.load_data(data_file)

    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt', encoding='utf-8') as f:
            for idx, block in enumerate(f.read().split('\n\n')):
                if not block.strip():
                    continue
                sentence, labels = '', []
                for i, item in enumerate(block.strip().split('\n')):
                    parts = item.split(' ')
                    if len(parts) != 2:
                        continue
                    char, tag = parts
                    sentence += char
                    if tag.startswith('B'):
                        labels.append([i, i, char, tag[2:]])
                    elif tag.startswith('I') and labels:
                        labels[-1][1] = i
                        labels[-1][2] += char
                if sentence:
                    Data[idx] = {'sentence': sentence, 'labels': labels}
        return Data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

print("\n✓ PeopleDaily 数据集类定义成功")

## 第四步：加载分词器和数据

In [ ]:
# ============ 模型选择配置 ============
# 1. 'bert-base-chinese'             - Google BERT（中文，推荐）⭐⭐⭐ (默认)
# 2. 'hfl/chinese-roberta-wwm-ext'   - 哈工大 RoBERTa（效果更好）⭐⭐⭐
# 3. 'hfl/chinese-macbert-base'      - MacBERT（推荐）⭐⭐⭐
#
MODEL_CHOICE = 'bert-base-chinese'  # ⬅️ 修改这里选择模型
model_name   = MODEL_CHOICE

print(f"✓ 模型选择: {model_name}")
print("\n加载分词器...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("✓ 分词器加载成功")

print("\n加载数据集...")
train_dataset = PeopleDaily(TRAIN_FILE)
dev_dataset   = PeopleDaily(DEV_FILE)
test_dataset  = PeopleDaily(TEST_FILE)
print(f"✓ 训练集: {len(train_dataset)} 条")
print(f"✓ 验证集: {len(dev_dataset)} 条")
print(f"✓ 测试集: {len(test_dataset)} 条")

## 第五步：查看数据样本

In [ ]:
for i in range(3):
    sample = train_dataset[i]
    print(f"\n=== 样本 {i+1} ===")
    sent_preview = sample['sentence'][:50]
    print(f"句子: {sent_preview}{'...' if len(sample['sentence']) > 50 else ''}")
    print(f"实体: {sample['labels'][:5]}{'...' if len(sample['labels']) > 5 else ''}")

# 实体分布统计
all_tags = [tag for i in range(len(train_dataset))
            for _, _, _, tag in train_dataset[i]['labels']]
tag_counter = Counter(all_tags)
print("\n训练集实体类型分布:")
for tag, count in sorted(tag_counter.items()):
    print(f"  {tag}: {count} 个")

## 第六步：超参数配置

In [ ]:
class Args:
    max_seq_length   = 512
    batch_size       = 4
    num_train_epochs = 3
    learning_rate    = 1e-5
    warmup_proportion= 0.1
    weight_decay     = 0.01
    adam_beta1       = 0.9
    adam_beta2       = 0.98
    adam_epsilon     = 1e-8
    seed             = 42
    output_dir       = OUTPUT_DIR
    id2label         = id2label
    label2id         = label2id
    num_labels       = len(id2label)

args = Args()
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)

if torch.backends.mps.is_available():
    args.device = torch.device('mps')
elif torch.cuda.is_available():
    args.device = torch.device('cuda')
else:
    args.device = torch.device('cpu')

print("超参数配置:")
for k, v in vars(args).items():
    if k not in ('id2label', 'label2id'):
        print(f"  {k}: {v}")

## 第七步：定义模型

**NER 模型结构**（Token Classification）：
```
输入: [CLS] 毛 泽 东 在 北 京 [SEP]
  ↓
BERT Encoder → [batch, seq_len, hidden_size]
  ↓
Dropout
  ↓
Linear(hidden → num_labels=7) → [batch, seq_len, 7]
  ↓
argmax → 每个 token 的 BIO 标签
```
**损失**：CrossEntropyLoss，只对非 padding token 计算（利用 attention_mask 过滤）。

In [ ]:
class BertForNER(BertPreTrainedModel):
    """基于 BERT 的命名实体识别模型（Token Classification）"""

    def __init__(self, config, args):
        super().__init__(config)
        self.num_labels = args.num_labels
        self.bert       = BertModel(config, add_pooling_layer=False)
        self.dropout    = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, self.num_labels)
        self.post_init()

    def forward(self, batch_inputs, labels=None):
        bert_output     = self.bert(**batch_inputs)
        sequence_output = self.dropout(bert_output.last_hidden_state)  # [B, seq_len, H]
        logits          = self.classifier(sequence_output)              # [B, seq_len, num_labels]

        loss = None
        if labels is not None:
            loss_fct  = CrossEntropyLoss()
            attn_mask = batch_inputs.get('attention_mask')
            if attn_mask is not None:
                # 只对非 padding token 计算 loss
                active_loss   = attn_mask.view(-1) == 1
                active_logits = logits.view(-1, self.num_labels)[active_loss]
                active_labels = labels.view(-1)[active_loss]
                loss = loss_fct(active_logits, active_labels)
            else:
                loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return loss, logits

print("✓ BertForNER 类定义成功")

## 第八步：创建 DataLoader

**关键**：将实体的**字符位置** → **token 位置**（使用 `char_to_token`），构造 token 级 BIO 标签序列。

In [ ]:
def to_device(args, batch_data):
    new_batch_data = {}
    for k, v in batch_data.items():
        if k == 'batch_inputs':
            new_batch_data[k] = {k_: v_.to(args.device) for k_, v_ in v.items()}
        else:
            new_batch_data[k] = torch.tensor(v).to(args.device)
    return new_batch_data


def get_dataLoader(args, dataset, tokenizer, shuffle=False):
    """创建 NER DataLoader，将字符级实体标注转换为 token 级 BIO 标签"""

    def collote_fn(batch_samples):
        batch_sentence, batch_labels = [], []
        for sample in batch_samples:
            batch_sentence.append(sample['sentence'])
            batch_labels.append(sample['labels'])

        batch_inputs = tokenizer(
            batch_sentence,
            max_length=args.max_seq_length,
            padding=True,
            truncation=True,
            return_tensors='pt'
        )

        # token 级标签（初始全部为 O=0）
        batch_label = np.zeros(batch_inputs['input_ids'].shape, dtype=int)

        for s_idx, sentence in enumerate(batch_sentence):
            # 单独对该句子分词，获取 char_to_token 映射
            encoding = tokenizer(sentence, max_length=args.max_seq_length, truncation=True)
            for char_start, char_end, _, tag in batch_labels[s_idx]:
                token_start = encoding.char_to_token(char_start)
                token_end   = encoding.char_to_token(char_end)
                if token_start is None or token_end is None:
                    continue
                batch_label[s_idx][token_start]            = args.label2id[f'B-{tag}']
                batch_label[s_idx][token_start+1:token_end+1] = args.label2id[f'I-{tag}']

        return {'batch_inputs': batch_inputs, 'labels': batch_label}

    return DataLoader(dataset, batch_size=args.batch_size, shuffle=shuffle, collate_fn=collote_fn)


print("创建 DataLoader...")
train_dataloader = get_dataLoader(args, train_dataset, tokenizer, shuffle=True)
dev_dataloader   = get_dataLoader(args, dev_dataset,   tokenizer, shuffle=False)
test_dataloader  = get_dataLoader(args, test_dataset,  tokenizer, shuffle=False)

print(f"✓ 训练集批次数: {len(train_dataloader)}")
print(f"✓ 验证集批次数: {len(dev_dataloader)}")
print(f"✓ 测试集批次数: {len(test_dataloader)}")

## 第九步：加载模型

In [ ]:
print(f"加载 BERT NER 模型: {model_name}...")
config = AutoConfig.from_pretrained(model_name)
model  = BertForNER.from_pretrained(model_name, config=config, args=args)
model  = model.to(args.device)

print(f"✓ 模型加载成功")
print(f"✓ 模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"✓ 模型所在设备: {next(model.parameters()).device}")

## 第十步：设置优化器和学习率调度器

In [ ]:
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': args.weight_decay},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0}
]

t_total = len(train_dataloader) * args.num_train_epochs
args.warmup_steps = int(t_total * args.warmup_proportion)

optimizer = AdamW(
    optimizer_grouped_parameters,
    lr=args.learning_rate,
    betas=(args.adam_beta1, args.adam_beta2),
    eps=args.adam_epsilon
)
lr_scheduler = get_scheduler(
    'linear', optimizer,
    num_warmup_steps=args.warmup_steps,
    num_training_steps=t_total
)

print(f"优化器: AdamW，学习率: {args.learning_rate}")
print(f"调度器: Linear Warmup，总步数: {t_total}，预热步数: {args.warmup_steps}")

## 第十一步：定义训练和评估函数

In [ ]:
def train_loop(args, dataloader, model, optimizer, lr_scheduler, epoch, total_loss):
    """训练一个 epoch"""
    model.train()
    finish_step_num = epoch * len(dataloader)
    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch+1} Training')

    for step, batch_data in enumerate(progress_bar, start=1):
        batch_data = to_device(args, batch_data)
        outputs    = model(**batch_data)
        loss       = outputs[0]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'avg_loss': f'{total_loss / (finish_step_num + step):.4f}'})

    return total_loss


def test_loop(args, dataloader, model):
    """
    评估：用 seqeval 计算实体级别的 F1（strict 模式）

    注意：
    - 跳过 [CLS]（idx=0）和 [SEP]（idx=seq_len-1）token
    - attention_mask 确定各句的真实长度（去除 padding）
    """
    true_labels, true_predictions = [], []

    model.eval()
    with torch.no_grad():
        for batch_data in tqdm(dataloader, desc='Evaluating'):
            batch_data  = to_device(args, batch_data)
            outputs     = model(**batch_data)
            logits      = outputs[1]

            predictions = logits.argmax(dim=-1).cpu().numpy()
            labels      = batch_data['labels'].cpu().numpy()
            seq_lens    = batch_data['batch_inputs']['attention_mask'].cpu().numpy().sum(axis=-1)

            true_labels += [
                [args.id2label[int(l)] for idx, l in enumerate(label) if 0 < idx < seq_len - 1]
                for label, seq_len in zip(labels, seq_lens)
            ]
            true_predictions += [
                [args.id2label[int(p)] for idx, p in enumerate(pred) if 0 < idx < seq_len - 1]
                for pred, seq_len in zip(predictions, seq_lens)
            ]

    return classification_report(true_labels, true_predictions, mode='strict', scheme=IOB2, output_dict=True)

print("✓ train_loop / test_loop 定义成功")

## 第十二步：开始训练 ⭐⭐⭐

⚠️ **这一步需要较长时间**（约 1-2 小时，GPU）

最佳模型按**加权 F1（weighted F1）** 选择。

In [ ]:
training_history = {
    'train_loss':        [],
    'dev_micro_f1':      [],
    'dev_macro_f1':      [],
    'dev_weighted_f1':   []
}

best_f1         = 0.0
best_model_path = os.path.join(OUTPUT_DIR, 'best_model.bin')
total_loss      = 0.0

print("开始训练...\n")
for epoch in range(args.num_train_epochs):
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{args.num_train_epochs}")
    print(f"{'='*60}")

    total_loss = train_loop(args, train_dataloader, model, optimizer, lr_scheduler, epoch, total_loss)
    avg_loss   = total_loss / ((epoch + 1) * len(train_dataloader))
    print(f"\n平均训练损失: {avg_loss:.4f}")
    training_history['train_loss'].append(avg_loss)

    metrics     = test_loop(args, dev_dataloader, model)
    micro_f1    = metrics['micro avg']['f1-score']
    macro_f1    = metrics['macro avg']['f1-score']
    weighted_f1 = metrics['weighted avg']['f1-score']

    print(f"\n验证集指标:")
    print(f"  micro F1:    {100*micro_f1:.2f}")
    print(f"  macro F1:    {100*macro_f1:.2f}")
    print(f"  weighted F1: {100*weighted_f1:.2f}")
    for cat in CATEGORIES:
        if cat in metrics:
            m = metrics[cat]
            print(f"  {cat}: P={100*m['precision']:.2f}  R={100*m['recall']:.2f}  F1={100*m['f1-score']:.2f}")

    training_history['dev_micro_f1'].append(100 * micro_f1)
    training_history['dev_macro_f1'].append(100 * macro_f1)
    training_history['dev_weighted_f1'].append(100 * weighted_f1)

    if weighted_f1 > best_f1:
        best_f1 = weighted_f1
        torch.save(model.state_dict(), best_model_path)
        print(f"\n✓ 保存最佳模型 (weighted F1: {100*best_f1:.2f})")

print("\n" + "="*60)
print("✅ 训练完成！")
print("="*60)

## 第十三步：绘制训练曲线

In [ ]:
epochs = range(1, args.num_train_epochs + 1)
fig, (ax_loss, ax_f1) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('模型训练收敛曲线', fontsize=15, fontweight='bold')

ax_loss.plot(epochs, training_history['train_loss'], marker='o', linewidth=2, markersize=8)
ax_loss.set_xlabel('Epoch', fontsize=12)
ax_loss.set_ylabel('Loss', fontsize=12)
ax_loss.set_title('训练 Loss 曲线', fontsize=13, fontweight='bold')
ax_loss.set_xticks(epochs)
ax_loss.grid(True, alpha=0.3)

ax_f1.plot(epochs, training_history['dev_micro_f1'],    label='micro F1',    marker='o', linewidth=2)
ax_f1.plot(epochs, training_history['dev_macro_f1'],    label='macro F1',    marker='s', linewidth=2)
ax_f1.plot(epochs, training_history['dev_weighted_f1'], label='weighted F1', marker='^', linewidth=2)
ax_f1.set_xlabel('Epoch', fontsize=12)
ax_f1.set_ylabel('F1 分数', fontsize=12)
ax_f1.set_title('验证集 F1 曲线', fontsize=13, fontweight='bold')
ax_f1.set_xticks(epochs)
ax_f1.legend(fontsize=11)
ax_f1.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ 曲线已保存")

## 第十四步：加载最佳模型和预测函数

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=args.device))
model.eval()
print(f"✓ 加载最佳模型: {best_model_path}")


def predict_entities(sentence, model, tokenizer, args):
    """
    对单个句子进行 NER 预测

    返回:
    [
      {"entity_group": "PER", "word": "毛泽东", "start": 0, "end": 3, "score": 0.98},
      ...
    ]
    """
    inputs  = tokenizer(
        sentence,
        max_length=args.max_seq_length,
        truncation=True,
        return_tensors='pt',
        return_offsets_mapping=True
    )
    offsets = inputs.pop('offset_mapping').squeeze(0)
    inputs  = to_device(args, {'batch_inputs': inputs})

    with torch.no_grad():
        outputs      = model(**inputs)
        logits       = outputs[1]
    probabilities = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    predictions   = logits.argmax(dim=-1)[0].cpu().numpy()

    entities = []
    idx = 1  # 跳过 [CLS]
    while idx < len(predictions) - 1:  # 跳过 [SEP]
        label = args.id2label[predictions[idx]]
        if label.startswith('B-'):
            etype  = label[2:]
            start  = offsets[idx][0].item()
            end    = offsets[idx][1].item()
            scores = [probabilities[idx][predictions[idx]]]
            while (idx + 1 < len(predictions) - 1 and
                   args.id2label[predictions[idx+1]] == f'I-{etype}'):
                idx  += 1
                end   = offsets[idx][1].item()
                scores.append(probabilities[idx][predictions[idx]])
            entities.append({
                'entity_group': etype,
                'word':  sentence[start:end],
                'start': start,
                'end':   end,
                'score': float(np.mean(scores))
            })
        idx += 1
    return entities

print("✓ predict_entities 函数定义成功")

## 第十五步：测试集预测示例

In [ ]:
print("\n" + "="*80)
print("测试集预测示例（前 5 条）")
print("="*80)

for i in range(min(5, len(test_dataset))):
    sample    = test_dataset[i]
    pred_ents = predict_entities(sample['sentence'], model, tokenizer, args)
    sent      = sample['sentence'][:60]
    print(f"\n示例 {i+1}:")
    print(f"句子: {sent}{'...' if len(sample['sentence']) > 60 else ''}")
    print(f"✓ 真实实体: {sample['labels'][:3]}")
    print(f"🤖 预测实体: {pred_ents[:3]}")
    print("-" * 80)

## 第十六步：自定义输入预测

In [ ]:
custom_sentences = [
    "毛泽东主席在北京天安门城楼上宣布中华人民共和国成立。",
    "阿里巴巴集团创始人马云在杭州创立了这家互联网公司。",
    "中国科学院和北京大学联合开展了这项量子计算研究。",
]

label_map = {'PER': '人名', 'LOC': '地名', 'ORG': '机构'}

print("\n" + "="*80)
print("自定义输入 NER 预测")
print("="*80)

for idx, sentence in enumerate(custom_sentences):
    entities = predict_entities(sentence, model, tokenizer, args)
    print(f"\n示例 {idx+1}: {sentence}")
    if entities:
        for ent in entities:
            print(f"  [{label_map.get(ent['entity_group'], ent['entity_group'])}] "
                  f"'{ent['word']}' (位置: {ent['start']}-{ent['end']}, 置信度: {ent['score']:.3f})")
    else:
        print("  （未识别到实体）")

## 第十七步：测试集最终评估

In [ ]:
print("在测试集上评估...")
test_metrics = test_loop(args, test_dataloader, model)

print("\n测试集详细报告:")
print(f"  {'类别':15s}  {'P':>6}  {'R':>6}  {'F1':>6}")
print(f"  {'─'*40}")
for key, val in test_metrics.items():
    if isinstance(val, dict) and 'f1-score' in val:
        p, r, f = val['precision'], val['recall'], val['f1-score']
        print(f"  {key:15s}  {100*p:6.2f}  {100*r:6.2f}  {100*f:6.2f}")

## 第十八步：模型保存和总结

In [ ]:
model_save_path = os.path.join(OUTPUT_DIR, 'final_model')
os.makedirs(model_save_path, exist_ok=True)
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"✓ 模型已保存到: {model_save_path}")

with open(os.path.join(OUTPUT_DIR, 'label_map.json'), 'w', encoding='utf-8') as f:
    json.dump({'id2label': id2label, 'label2id': label2id}, f, indent=2, ensure_ascii=False)
with open(os.path.join(OUTPUT_DIR, 'training_history.json'), 'w', encoding='utf-8') as f:
    json.dump(training_history, f, indent=2)

print("\n" + "="*80)
print("✅ 项目总结")
print("="*80)
print(f"""
【模型信息】
  架构: BERT-Base-Chinese（Token Classification）
  任务: 命名实体识别 NER（LOC / ORG / PER）
  标注方案: IOB2，共 {len(id2label)} 个标签
  设备: {args.device}

【最终指标（测试集）】
  micro F1:    {100*test_metrics['micro avg']['f1-score']:.2f}
  macro F1:    {100*test_metrics['macro avg']['f1-score']:.2f}
  weighted F1: {100*test_metrics['weighted avg']['f1-score']:.2f}

【下一步】
  1. 尝试更强的预训练模型: hfl/chinese-roberta-wwm-ext
  2. 添加 CRF 层利用标签转移约束（参考 run_ner_crf.py）
  3. 加载保存的模型:
     config    = AutoConfig.from_pretrained('{model_save_path}')
     tokenizer = AutoTokenizer.from_pretrained('{model_save_path}')
     model     = BertForNER.from_pretrained('{model_save_path}', config=config, args=args)
""")
print("="*80)